In [0]:
%pip install langgraph langchain-core mlflow tiktoken requests --quiet


In [0]:
%restart_python

In [0]:
import mlflow

mlflow.set_experiment("/Users/sandeep.b@diggibyte.com/langgraph-mock-demo")
mlflow.langchain.autolog()

# Then re-run app.invoke(...) and check the run in Experiments → Traces.

In [0]:
from typing import List, Dict, Any

def mock_llm(messages: List[Dict[str, str]], max_tokens: int = 256, temperature: float = 0.2) -> str:
    """
    Mock LLM: returns a canned answer based on the last user message.
    Replace this with a real LLM call when your workspace supports it.
    """
    last_user = None
    for m in reversed(messages):
        if m["role"] == "user":
            last_user = m["content"]
            break

    text = (last_user or "").lower()

    if "cdc" in text and "change data capture" in text:
        return (
            "Change Data Capture (CDC) captures only changed data in source tables "
            "and streams those changes to downstream systems. It reduces load compared "
            "to full table loads and enables near real-time updates in data lakes and warehouses."
        )
    elif "scd" in text:
        return (
            "SCD (Slowly Changing Dimension) tracks historical changes in dimension tables. "
            "Type 2 keeps history by adding new rows with versioning and valid-from/to timestamps."
        )
    elif "delta" in text:
        return (
            "Delta Lake is an open-source storage layer that brings ACID transactions, "
            "schema enforcement, and time travel to data lakes, often used with Spark and Databricks."
        )
    else:
        return (
            "This is a mock LLM response. In production, this would call a real model "
            "like databricks-meta-llama-3-3-70b-instruct via Databricks Foundation Model APIs."
        )

In [0]:
from typing import Annotated, Sequence
from typing_extensions import TypedDict
import operator

class AgentState(TypedDict):
    messages: Annotated[Sequence[str], operator.add]
    answer: str
    remaining_tokens: int
    iteration_count: int

In [0]:
def llm_node(state: AgentState):
    if state["remaining_tokens"] < 100 or state["iteration_count"] >= 5:
        return {
            "messages": ["[Budget] Token or iteration limit reached."],
            "answer": "Budget exhausted."
        }

    user_msg = state["messages"][-1] if state["messages"] else "Hello"
    messages = [
        {"role": "system", "content": "You are a helpful assistant. Answer briefly in 2–3 sentences."},
        {"role": "user", "content": user_msg}
    ]

    answer = mock_llm(messages, max_tokens=200, temperature=0.2)

    # Rough token estimate (1 token ≈ 4 chars)
    used = max(50, len(answer) // 4)

    return {
        "messages": [f"Assistant: {answer}"],
        "answer": answer,
        "remaining_tokens": state["remaining_tokens"] - used,
        "iteration_count": state["iteration_count"] + 1
    }

def guard_node(state: AgentState):
    if len(state.get("answer", "")) > 180:
        return {"messages": ["[Guard] Answer is long; consider summarizing."]}
    return {}

In [0]:
from langgraph.graph import StateGraph, START, END

graph = StateGraph(AgentState)
graph.add_node("llm", llm_node)
graph.add_node("guard", guard_node)

graph.add_edge(START, "llm")
graph.add_edge("llm", "guard")
graph.add_edge("guard", END)

app = graph.compile()

In [0]:
initial_state: AgentState = {
    "messages": ["Explain CDC (Change Data Capture) in data engineering in 2–3 sentences."],
    "answer": "",
    "remaining_tokens": 1000,
    "iteration_count": 0
}

result = app.invoke(initial_state)
print("Final answer:", result["answer"])
print("Remaining tokens:", result["remaining_tokens"])
print("Iterations:", result["iteration_count"])

In [0]:
from typing import Annotated, Sequence, Literal
from typing_extensions import TypedDict
import operator

class AgentState(TypedDict):
    messages: Annotated[Sequence[str], operator.add]
    answer: str
    remaining_tokens: int
    iteration_count: int
    next_step: Literal["research", "explain", "final"]

In [0]:
MAX_ITERATIONS = 3

def supervisor(state: AgentState):
    # Stop if we’ve looped too many times
    if state["iteration_count"] >= MAX_ITERATIONS:
        return {"next_step": "final"}

    text = " ".join(state["messages"]).lower()

    # If we already have a final-style answer, stop
    if "final answer" in text or "mock final" in text:
        return {"next_step": "final"}

    if "compare" in text or "vs" in text:
        return {"next_step": "research"}
    elif "explain" in text or "what is" in text:
        return {"next_step": "explain"}
    else:
        return {"next_step": "final"}

In [0]:
def research_node(state: AgentState):
    answer = "Mock research: CDC is often implemented using log-based or trigger-based mechanisms."
    new_messages = [f"Research: {answer}"]
    return {
        "messages": new_messages,
        "answer": answer,
        "iteration_count": state["iteration_count"] + 1
    }

def explain_node(state: AgentState):
    answer = "Mock explain: CDC captures only changed rows and streams them to targets."
    new_messages = [f"Explain: {answer}"]
    return {
        "messages": new_messages,
        "answer": answer,
        "iteration_count": state["iteration_count"] + 1
    }

def final_node(state: AgentState):
    answer = "Mock final answer: CDC + SCD patterns are key for incremental data pipelines."
    new_messages = [f"Final: {answer}"]
    return {
        "messages": new_messages,
        "answer": answer,
        "iteration_count": state["iteration_count"] + 1
    }

In [0]:
from langgraph.graph import StateGraph, START, END

graph2 = StateGraph(AgentState)
graph2.add_node("supervisor", supervisor)
graph2.add_node("research", research_node)
graph2.add_node("explain", explain_node)
graph2.add_node("final", final_node)

graph2.add_edge(START, "supervisor")

def route_supervisor(state: AgentState):
    return state["next_step"]

graph2.add_conditional_edges(
    "supervisor",
    route_supervisor,
    {
        "research": "research",
        "explain": "explain",
        "final": "final"
    }
)

# Sub-nodes go back to supervisor (loop), but supervisor will eventually choose "final"
graph2.add_edge("research", "supervisor")
graph2.add_edge("explain", "supervisor")
graph2.add_edge("final", END)

app2 = graph2.compile()

In [0]:
initial_state2: AgentState = {
    "messages": ["Explain CDC briefly."],
    "answer": "",
    "remaining_tokens": 1000,
    "iteration_count": 0,
    "next_step": ""
}

result2 = app2.invoke(initial_state2)
print("Supervisor answer:", result2["answer"])
print("Iterations:", result2["iteration_count"])